# NPH-Client — Guia Rápido (v1.0.0)

Exemplo oficial de consulta de estações, medições, previsões e visualização com `nph-client`.

Antes de executar, defina sua chave na variável de ambiente `NPH_API_KEY` (ou substitua no bloco abaixo apenas localmente, sem versionar sua chave).

In [ ]:
import os
from datetime import timedelta
import matplotlib.pyplot as plt
from nph_client import NPHClient
from nph_client.plotting import AxisConfig, Threshold, plot_series

api_key = os.getenv("NPH_API_KEY", "SUA_API_KEY_AQUI")
client = NPHClient(api_key)

In [ ]:
# 1. Listar estações disponíveis
stations_df = client.get_stations()
stations_df.head()

In [ ]:
# 2. Consultar medições e previsões (Ex: Ilha das Palmas - ID 351)
start_date = "2025-12-12"
end_date = "2025-12-13"
    
df_measurements = client.get_measurements(
    stations=[351],
    start_date=start_date,
    end_date=end_date,
    return_value=0,
    time_offset=timedelta(hours=-3),
)

df_forecasts = client.get_forecasts(
    stations=[351],
    model_id=4,       # MOHID
    param_id=13,      # Nível do Mar
    start_date=start_date,
    end_date=end_date,
)
    
df_measurements.head()

In [ ]:
# 3. Plotar comparação com limiares de atenção e alerta
limite_atencao = Threshold(
    value=1.8,
    color="orange",
    label="ESTADO DE ATENÇÃO",
    fill_above=True,
    alpha=0.15,
)

limite_alerta = Threshold(
    value=2.0,
    color="red",
    label="ESTADO DE ALERTA",
    fill_above=True,
)

conf_measurement = AxisConfig(
    plot_type="scatter",
    label="Nível do Mar (m)",
    columns=list(df_measurements.columns),
    min_val=-0.5,
    max_val=2.5,
    step=0.5,
    thresholds=[limite_atencao, limite_alerta],
    plot_kwargs={"marker": "8", "s": 8, "alpha": 1, "color": "black"},
)

conf_forecast = AxisConfig(
    plot_type="line",
    label="Nível do Mar (m)",
    columns=list(df_forecasts.columns),
    min_val=-0.5,
    max_val=2.5,
    step=0.5,
    plot_kwargs={"alpha": 1, "linewidth": 1.5, "color": "green"},
)

plot_series(
    df=[df_measurements, df_forecasts],
    primary_conf=conf_measurement,
    secondary_conf=conf_forecast,
    title="Monitoramento de Ressaca - Comparação",
    x_lim=True,
)

plt.show()